In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from scipy.optimize import curve_fit

# 1. Load data and rebuild the continuous price model from Task 1
df = pd.read_csv("Nat_Gas.csv")
df['Dates'] = pd.to_datetime(df['Dates'], format='mixed')
df = df.sort_values('Dates')

min_date = df['Dates'].min()
df['Days'] = (df['Dates'] - min_date).dt.days

def price_model(t, amplitude, frequency, phase_shift, slope, intercept):
    return amplitude * np.sin(frequency * t + phase_shift) + slope * t + intercept

initial_guesses = [0.5, 2 * np.pi / 365.25, 0, 0.001, 10]
popt, _ = curve_fit(price_model, df['Days'], df['Prices'], p0=initial_guesses)

# The price estimation function from Task 1
def estimate_price(input_date_str):
    target_date = pd.to_datetime(input_date_str)
    days_since = (target_date - min_date).days
    return price_model(days_since, *popt)

In [4]:
def price_storage_contract(injection_dates, withdrawal_dates, injection_rates, withdrawal_rates, max_volume, storage_cost_per_month):
    """
    Prices a natural gas storage contract.
    
    Parameters:
    - injection_dates: list of dates (strings) when gas is bought and injected
    - withdrawal_dates: list of dates (strings) when gas is sold and withdrawn
    - injection_rates: list of volumes injected on each injection date
    - withdrawal_rates: list of volumes withdrawn on each withdrawal date
    - max_volume: maximum physical volume the storage facility can hold
    - storage_cost_per_month: cost per unit of volume stored per month
    """
    
    total_cash_flow = 0.0
    stored_inventory = 0.0
    
    # Track inventory and cash flows for injections (purchases)
    for date_str, volume in zip(injection_dates, injection_rates):
        if stored_inventory + volume > max_volume:
            raise ValueError(f"Injection at {date_str} exceeds maximum storage capacity!")
        
        price = estimate_price(date_str)
        cost = volume * price
        total_cash_flow -= cost  # Money flowing OUT to buy gas
        stored_inventory += volume

    # Track inventory and cash flows for withdrawals (sales)
    for date_str, volume in zip(withdrawal_dates, withdrawal_rates):
        if stored_inventory - volume < 0:
            raise ValueError(f"Withdrawal at {date_str} exceeds currently stored inventory!")
        
        price = estimate_price(date_str)
        revenue = volume * price
        total_cash_flow += revenue  # Money flowing IN from selling gas
        stored_inventory -= volume

    # Calculate storage duration and ongoing holding costs
    # For simplicity, we approximate total storage duration from the first injection to the last withdrawal
    first_inj = pd.to_datetime(min(injection_dates))
    last_with = pd.to_datetime(max(withdrawal_dates))
    total_months = (last_with - first_inj).days / 30.44  # Average days in a month
    
    total_storage_cost = max_volume * storage_cost_per_month * total_months
    total_cash_flow -= total_storage_cost  # Subtract storage fees

    return total_cash_flow

In [5]:
# Sample Test Inputs
inj_dates = ['2024-06-30']       # Summer injection (lower price)
with_dates = ['2024-12-31']      # Winter withdrawal (higher price)
inj_volumes = [100000]           # 100,000 units of gas
with_volumes = [100000]          # 100,000 units sold back out
max_cap = 150000                 # Facility max capacity
monthly_cost = 0.02              # $0.02 storage fee per unit per month

contract_value = price_storage_contract(
    injection_dates=inj_dates,
    withdrawal_dates=with_dates,
    injection_rates=inj_volumes,
    withdrawal_rates=with_volumes,
    max_volume=max_cap,
    storage_cost_per_month=monthly_cost
)

print(f"Estimated Net Value of the Storage Contract: ${contract_value:,.2f}")

Estimated Net Value of the Storage Contract: $130,834.69
